In [7]:
import numpy as np  # import numerical python
import matplotlib.pyplot as plt  # import plotting functions
import seaborn as sns  # import nicer plotting functions
import polars as pl  # import polars to import data
import tifffile as tiff
from tifffile import imwrite, imread
from copy import deepcopy
import os
import sys
import shutil

sys.path.append('../..')
from src import IOFunctions

IO = IOFunctions.IO_Functions()

from src import Multicolour_Simulation_Functions

MSF = Multicolour_Simulation_Functions.MultiC_Sim_Funcs()

from src import PSFFunctions

PSF = PSFFunctions.PSF_Functions()

from src import PlottingFunctions

plotter = PlottingFunctions.Plotter()

from src import ImageAnalysisFunctions

I_AF = ImageAnalysisFunctions.Image_Analysis_Functions()

from src import sCMOSFunctions

sCMOS = sCMOSFunctions.sCMOS_Functions()

from src import SpectralFunctions

S_F = SpectralFunctions.Spectral_Funcs()

from src import MaskFunctions

M_F = MaskFunctions.Mask_Functions()

from src import PSFFunctions

PSF = PSFFunctions.PSF_Functions()
from copy import copy

def gaussian_model(x, mu, sigma):
    y = np.exp(-0.5*(x-mu)**2 / sigma**2)
    return y/np.max(y)

In [8]:
from src import IOFunctions

IO = IOFunctions.IO_Functions()

In [9]:
R, G, B, wavelength = S_F.getpixelefficiency('../../Spectra/Camera_QE/CS505CU_QE.csv')

In [10]:
cameras = ['Camera_Normal', 'Camera_Sharp', 'Camera_Broad']

R_nonsharp = np.max(G)*gaussian_model(wavelength, 650, 500)
G_nonsharp = np.max(G)*gaussian_model(wavelength, 650, 500)
B_nonsharp = np.max(G)*gaussian_model(wavelength, 650, 500)

R_sharp = copy(R)
G_sharp = copy(G)
B_sharp = copy(B)
B_sharp[wavelength < 500] = np.max(B)
B_sharp[wavelength >= 500] = 0
G_sharp[(wavelength < 500) | (wavelength > 600)] = 0
G_sharp[(wavelength < 600) & (wavelength >= 500)] = np.max(B)
R_sharp[wavelength < 600] = 0
R_sharp[wavelength >= 600] = np.max(B)

wavelength = wavelength
pixel_QYs_normal = np.vstack([B, G, R])
pixel_QYs_broad = np.vstack([B_nonsharp, G_nonsharp, R_nonsharp])
pixel_QYs_sharp = np.vstack([B_sharp, G_sharp, R_sharp])

In [11]:
# values of phootns are per 100 ms from Steen, P. R. et al. The DNA-PAINT palette: a comprehensive performance analysis of fluorescent dyes. Nat Methods 21, 1755–1762 (2024).
# or from Dempsey, G. T., Vaughan, J. C., Chen, K. H., Bates, M. & Zhuang, X. Evaluation of fluorophores for optimal performance in localization-based super-resolution imaging. Nat Methods 8, 1027–1036 (2011).
single_molecule_dyes = np.array([['ATTO 488', 2073, 0.8],
                                 ['Alexa Fluor 488', 2811, 0.92],
                                 ['Abberior Star 488', 2969, 0.89],
                                 ['CF488A', 3879, 0.9],
                                 ['Cy3B', 23195, 0.58],
                                 ['ATTO 565', 11600, 0.9],
                                 ['Janelia Fluor JF585-HaloTag conjugate', 2429, 0.78],
                                 ['CF568', 13388, 0.9],
                                 ['Cy5', 7801, 0.27],
                                 ['Cy5B', 17090, 0.4],
                                 ['ATTO 643', 23327, 0.62],
                                 ['ATTO 647N', 18448, 0.65],
                                 ['ATTO 655', 8273, 0.3],
                                 ['CF640R', 12024, 0.3],
                                 ['CF660R', 10399, 0.3],
                                 ['abberior STAR 635', 12731, 0.9],
                                 ['Janelia Fluor JF646-HaloTag conjugate', 14440, 0.54],
                                 ['Alexa Fluor 647', 10348, 0.33],
                                 ['Cy2', 6241, 0.12],
                                 ['Cy3', 11022, 0.15],
                                 ['Tetramethylrhodamine (TAMRA, TRITC)', 4884, 0.2],
                                 ['Cy3.5', 4968, 0.15],
                                 ['ATTO 647', 1526, 0.2],
                                 ['ATTO 680', 1656, 0.3],
                                 ['Cy5.5', 6337, 0.28],
                                 ['Cy7', 852, 0.28],
                                 ['Alexa Fluor 750', 703, 0.12],
                                 ['ATTO 740', 779, 0.1],
                                 ['Alexa Fluor 790', 740, 0.1]], dtype='object')


In [12]:
single_molecule_dataset = pl.DataFrame(data=single_molecule_dyes, schema=['dye', 'photons', 'QY'])

In [14]:
corrected_photons_normal = np.zeros(len(single_molecule_dyes))
corrected_photons_bayer = np.zeros(len(single_molecule_dyes))
corrected_photons_sharp = np.zeros(len(single_molecule_dyes))

In [17]:
for dyeval in np.arange(len(single_molecule_dataset)):
    dye = single_molecule_dataset['dye'].to_numpy()[dyeval]
    if dye == 'Abberior Star 488':
        dyestr = 'Alexa Fluor 488'
    else:
        dyestr = dye
    photonval = single_molecule_dataset['photons'].to_numpy()[dyeval]
    _, efficiency = S_F.get_pixel_fractions_dye_and_filters(dyes=dyestr, filters=[], wavelength=wavelength,
                                                            pixel_QYs=np.vstack([R, G, G, B]))
    corrected_photons_bayer[dyeval] = np.mean(photonval * efficiency)
    _, efficiency_sharp = S_F.get_pixel_fractions_dye_and_filters(dyes=dyestr, filters=[], wavelength=wavelength, 
                                                            pixel_QYs=np.vstack([R_sharp, G_sharp, G_sharp, B_sharp]))
    corrected_photons_sharp[dyeval] = np.mean(photonval * efficiency_sharp)
    _, efficiency_broad = S_F.get_pixel_fractions_dye_and_filters(dyes=dyestr, filters=[], wavelength=wavelength, 
                                                            pixel_QYs=np.vstack([G_nonsharp, G_nonsharp, G_nonsharp, G_nonsharp]))
    corrected_photons_normal[dyeval] = np.mean(photonval * efficiency_broad)

In [9]:
columns = ['dye', 'photons', 'QY', 'corrected_photons_bayer', 'corrected_photons_sharp', 'corrected_photons_normal',
           'percentage_normal_camera', 'percentage_bayer_camera', 'percentage_sharp_camera']
data = {}
data['dye'] = single_molecule_dyes[:, 0]
data['photons'] = np.asarray(single_molecule_dyes[:, 1], dtype=np.float32)
data['QY'] = np.asarray(single_molecule_dyes[:, 2], dtype=np.float32)
data['corrected_photons'] = np.asarray(corrected_photons, dtype=np.float32)
single_molecule_dataset = pl.DataFrame(data=data)

In [10]:
single_molecule_dataset.write_csv('Photons_Per_Dye.csv')

In [11]:
limit_xc_loose = 10
limit_colour_loose = 0.1
limit_xc_tight = 5
limit_colour_tight = 0.05

In [12]:
#data_folder = "/home/jbeckwith/Documents/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Data/Salix/CS505CU_Calibration/"
data_folder = '../../Camera_Calibrations/Ximea_Camera/'
gain = IO.read_tiff(os.path.join(data_folder, "gain.tif"))
offset = IO.read_tiff(os.path.join(data_folder, "offset.tif"))
variance = IO.read_tiff(os.path.join(data_folder, "variance.tif"))
readnoise = IO.read_tiff(os.path.join(data_folder, "readnoise.tif"))
rqe = IO.read_tiff(os.path.join(data_folder, "rqe.tif"))

In [13]:
image_size = 20
masks = M_F.get_masks(size_x=image_size, size_y=image_size)
wavelength = wavelength
pixel_QYs = np.vstack([B, G, R])
camera_parameters = {}
camera_parameters["gain"] = np.full((image_size, image_size), np.median(gain))
camera_parameters["variance"] = np.full((image_size, image_size), np.median(variance))
camera_parameters["readnoise"] = np.full((image_size, image_size), np.median(readnoise))
camera_parameters["offset"] = np.full((image_size, image_size), np.median(offset))
camera_parameters["rqe"] = np.full((image_size, image_size), np.median(rqe))
camera_parameters["pixel_QYs"] = pixel_QYs
camera_parameters["pixel_order"] = ['B', 'G', 'R']
camera_parameters["pixel_order_indices"] = [0, 1, 2]
camera_parameters["masks"] = masks

In [14]:
n_bootstrap = 20000
background_photons = 40
pixel_size = 69
NA = 1.49

In [15]:
save_folder = r'/home/jbeckwith/Documents/pCloud/Chemistry/Lee/Data/Simulation/20250729_DyeSuitability'
if not os.path.isdir(save_folder):
    os.makedirs(save_folder)

In [16]:
import types
smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma" :  1.5}
smoothing_function.extent =  1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = "image"

In [17]:
xc_loose = np.zeros(len(single_molecule_dataset['dye'].to_numpy()))
xc_tight = np.zeros(len(single_molecule_dataset['dye'].to_numpy()))
colour_loose = np.zeros(len(single_molecule_dataset['dye'].to_numpy()))
colour_tight = np.zeros(len(single_molecule_dataset['dye'].to_numpy()))
xc_precision = np.zeros(len(single_molecule_dataset['dye'].to_numpy()))
colour_precision = np.zeros(len(single_molecule_dataset['dye'].to_numpy()))

In [18]:
for dyeval in np.arange(len(single_molecule_dataset)):
    dyevalue = single_molecule_dataset['dye'].to_numpy()[dyeval]
    if dyevalue == 'Abberior Star 488':
        dye = 'Alexa Fluor 488'
    else:
        dye = dyevalue
    photonvalue = single_molecule_dataset['photons'].to_numpy()[dyeval]
    print("Analysing dye {}".format(dye), end="\r",flush=True,)        
    MSF.test_fit_method(
                        dye,
                        [],
                        wavelength,
                        camera_parameters,
                        save_folder,
                        np.array([photonvalue]),
                        smoothing_function=smoothing_function,
                        starting_flag="testingdyesuitability_",
                        n_bootstrap=n_bootstrap,
                        background_photons=background_photons,
                        NA=NA,
                        pixel_size=pixel_size,
                        cpu_fraction=1,
                    )
    analysis_files = os.listdir(save_folder)
    mean_wrongness = [os.path.join(save_folder, x) for x in analysis_files if 'RMSE_mean' in x]
    data = pl.read_csv(mean_wrongness[0])
    loc_prec = 0.5*float(data['xc'].to_numpy()[0] + data['yc'].to_numpy()[0])
    colour_prec = float(data['colour_distance'].to_numpy()[0])
    xc_precision[dyeval] = loc_prec
    colour_precision[dyeval] = colour_prec
    if loc_prec < limit_xc_loose:
        xc_loose[dyeval] = 1
    if loc_prec < limit_xc_tight:
        xc_tight[dyeval] = 1
    if colour_prec < limit_colour_loose:
        colour_loose[dyeval] = 1
    if colour_prec < limit_colour_tight:
        colour_tight[dyeval] = 1
    for file in analysis_files:
        os.remove(os.path.join(save_folder, file))

LM fitting: 100%|████████████████████████| 1800/1800 [00:07<00:00, 228.38task/s]


LM fitting: 100%|████████████████████████| 1800/1800 [00:07<00:00, 231.34task/s]

LM fitting: 100%|████████████████████████| 1800/1800 [00:07<00:00, 230.83task/s]


LM fitting: 100%|████████████████████████| 1800/1800 [00:07<00:00, 228.32task/s]


LM fitting: 100%|████████████████████████| 1800/1800 [00:07<00:00, 234.58task/s]


...

LM fitting: 100%|████████████████████████| 1800/1800 [00:07<00:00, 228.11task/s]


LM fitting: 100%|████████████████████████| 1800/1800 [00:08<00:00, 224.17task/s]


LM fitting: 100%|████████████████████████| 1800/1800 [00:07<00:00, 228.42task/s]


LM fitting: 100%|████████████████████████| 1800/1800 [00:07<00:00, 230.28task/s]


LM fitting: 100%|████████████████████████| 1800/1800 [00:07<00:00, 227.58task/s]

In [35]:
columns = ['dye', 'photons', 'QY', 'corrected_photons', 'precision_10nm', 'precision_5nm', 'colour_10perc', 'colour_5perc']
data = {}
data['dye'] = single_molecule_dyes[:, 0]
data['photons'] = np.asarray(single_molecule_dyes[:, 1], dtype=np.float32)
data['QY'] = np.asarray(single_molecule_dyes[:, 2], dtype=np.float32)
data['corrected_photons'] = np.asarray(corrected_photons, dtype=np.float32)
data['precision_10nm'] = np.asarray(xc_loose, dtype=np.uint8)
data['precision_5nm'] = np.asarray(xc_tight, dtype=np.uint8)
data['colour_10perc'] = np.asarray(colour_loose, dtype=np.uint8)
data['colour_5perc'] = np.asarray(colour_tight, dtype=np.uint8)
data['xc_precision'] = xc_precision
data['colour_precision'] = colour_precision

single_molecule_dataset = pl.DataFrame(data=data)
single_molecule_dataset.write_csv('Photons_Per_Dye_BetterCamera_Calib.csv')

In [ ]:
R, G, B, wavelength = S_F.getpixelefficiency('../../Spectra/Camera_QE/CS505CU_QE.csv')

In [38]:
factor = 0.95 / np.max(G)

In [39]:
Rbetter = R*factor
Gbetter = G*factor
Bbetter = B*factor

In [40]:
image_size = 20
masks = M_F.get_masks(size_x=image_size, size_y=image_size)
wavelength = wavelength
pixel_QYs = np.vstack([Bbetter, Gbetter, Rbetter])
camera_parameters = {}
camera_parameters["gain"] = np.full((image_size, image_size), np.median(gain))
camera_parameters["variance"] = np.full((image_size, image_size), np.median(variance))
camera_parameters["readnoise"] = np.full((image_size, image_size), np.median(readnoise))
camera_parameters["offset"] = np.full((image_size, image_size), np.median(offset))
camera_parameters["rqe"] = np.full((image_size, image_size), np.median(rqe))
camera_parameters["pixel_QYs"] = pixel_QYs
camera_parameters["pixel_order"] = ['B', 'G', 'R']
camera_parameters["pixel_order_indices"] = [0, 1, 2]
camera_parameters["masks"] = masks

In [41]:
for dyeval in np.arange(len(single_molecule_dataset)):
    dye = single_molecule_dataset['dye'].to_numpy()[dyeval]
    if dye == 'Abberior Star 488':
        dyestr = 'Alexa Fluor 488'
    else:
        dyestr = dye
    photonval = single_molecule_dataset['photons'].to_numpy()[dyeval]
    _, efficiency = S_F.get_pixel_fractions_dye_and_filters(dyes=dyestr, filters=[], wavelength=wavelength, pixel_QYs=np.vstack([Rbetter, Gbetter, Gbetter, Bbetter]))
    corrected_photons[dyeval] = np.mean(photonval * efficiency)

In [43]:
columns = ['dye', 'photons', 'QY', 'corrected_photons']
data = {}
data['dye'] = single_molecule_dyes[:, 0]
data['photons'] = np.asarray(single_molecule_dyes[:, 1], dtype=np.float32)
data['QY'] = np.asarray(single_molecule_dyes[:, 2], dtype=np.float32)
data['corrected_photons'] = np.asarray(corrected_photons, dtype=np.float32)
single_molecule_dataset = pl.DataFrame(data=data)
single_molecule_dataset.write_csv('Photons_Per_Dye_BetterCamera.csv')

In [44]:
xc_loose = np.zeros(len(single_molecule_dataset['dye'].to_numpy()))
xc_tight = np.zeros(len(single_molecule_dataset['dye'].to_numpy()))
colour_loose = np.zeros(len(single_molecule_dataset['dye'].to_numpy()))
colour_tight = np.zeros(len(single_molecule_dataset['dye'].to_numpy()))
xc_precision = np.zeros(len(single_molecule_dataset['dye'].to_numpy()))
colour_precision = np.zeros(len(single_molecule_dataset['dye'].to_numpy()))

In [45]:
for dyeval in np.arange(len(single_molecule_dataset)):
    dyevalue = single_molecule_dataset['dye'].to_numpy()[dyeval]
    if dyevalue == 'Abberior Star 488':
        dye = 'Alexa Fluor 488'
    else:
        dye = dyevalue
    photonvalue = single_molecule_dataset['photons'].to_numpy()[dyeval]
    print("Analysing dye {}".format(dye), end="\r",flush=True,)        
    MSF.test_fit_method(
                        dye,
                        [],
                        wavelength,
                        camera_parameters,
                        save_folder,
                        np.array([photonvalue]),
                        smoothing_function=smoothing_function,
                        starting_flag="testingdyesuitability_",
                        n_bootstrap=n_bootstrap,
                        background_photons=background_photons,
                        NA=NA,
                        pixel_size=pixel_size,
                        cpu_fraction=1,
                    )
    analysis_files = os.listdir(save_folder)
    mean_wrongness = [os.path.join(save_folder, x) for x in analysis_files if 'RMSE_mean' in x]
    data = pl.read_csv(mean_wrongness[0])
    loc_prec = 0.5*float(data['xc'].to_numpy()[0] + data['yc'].to_numpy()[0])
    colour_prec = float(data['colour_distance'].to_numpy()[0])
    xc_precision[dyeval] = loc_prec
    colour_precision[dyeval] = colour_prec
    if loc_prec < limit_xc_loose:
        xc_loose[dyeval] = 1
    if loc_prec < limit_xc_tight:
        xc_tight[dyeval] = 1
    if colour_prec < limit_colour_loose:
        colour_loose[dyeval] = 1
    if colour_prec < limit_colour_tight:
        colour_tight[dyeval] = 1
    for file in analysis_files:
        os.remove(os.path.join(save_folder, file))

LM fitting: 100%|████████████████████████| 1800/1800 [00:07<00:00, 249.65task/s]


LM fitting: 100%|████████████████████████| 1800/1800 [00:07<00:00, 251.50task/s]


LM fitting: 100%|████████████████████████| 1800/1800 [00:07<00:00, 250.23task/s]


LM fitting: 100%|████████████████████████| 1800/1800 [00:07<00:00, 247.93task/s]

LM fitting: 100%|████████████████████████| 1800/1800 [00:07<00:00, 244.12task/s]

...


LM fitting: 100%|████████████████████████| 1800/1800 [00:07<00:00, 255.75task/s]


LM fitting: 100%|████████████████████████| 1800/1800 [00:07<00:00, 256.32task/s]

LM fitting: 100%|████████████████████████| 1800/1800 [00:06<00:00, 258.21task/s]


LM fitting: 100%|████████████████████████| 1800/1800 [00:06<00:00, 258.78task/s]


LM fitting: 100%|████████████████████████| 1800/1800 [00:06<00:00, 259.64task/s]

In [46]:
columns = ['dye', 'photons', 'QY', 'corrected_photons', 'precision_10nm', 'precision_5nm', 'colour_10perc', 'colour_5perc']
data = {}
data['dye'] = single_molecule_dyes[:, 0]
data['photons'] = np.asarray(single_molecule_dyes[:, 1], dtype=np.float32)
data['QY'] = np.asarray(single_molecule_dyes[:, 2], dtype=np.float32)
data['corrected_photons'] = np.asarray(corrected_photons, dtype=np.float32)
data['precision_10nm'] = np.asarray(xc_loose, dtype=np.uint8)
data['precision_5nm'] = np.asarray(xc_tight, dtype=np.uint8)
data['colour_10perc'] = np.asarray(colour_loose, dtype=np.uint8)
data['colour_5perc'] = np.asarray(colour_tight, dtype=np.uint8)
data['xc_precision'] = xc_precision
data['colour_precision'] = colour_precision

single_molecule_dataset = pl.DataFrame(data=data)
single_molecule_dataset.write_csv('Photons_Per_Dye_BetterCamera_Calib.csv')